# Heisenberg Analysis Demo

This notebook demonstrates the complete Heisenberg analysis pipeline, from loading data to interpreting results.

**Contents:**
1. Setup and imports
2. Creating synthetic test data
3. Basic tuningfork analysis
4. Interpreting results
5. Visualization
6. Iterative diffuse filtering
7. Parameter exploration

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Heisenberg imports
from heisenberg.core.tuningfork import TuningForkConfig, run_tuningfork
from heisenberg.core.diffuse_iteration import DiffuseIterationConfig, run_diffuse_iteration
from heisenberg.io import write_results, write_summary

# Configure matplotlib for nice plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

print("Imports successful!")

## 2. Creating Synthetic Test Data

We'll create synthetic star and gas maps with known properties to demonstrate the analysis. In real usage, you would load FITS files instead.

In [ ]:
def make_gaussian(shape, cy, cx, amp, sigma):
    """Create a 2D Gaussian peak."""
    y, x = np.ogrid[:shape[0], :shape[1]]
    return amp * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * sigma**2))

def create_synthetic_galaxy(shape=(256, 256), n_peaks=20, seed=42):
    """
    Create synthetic star and gas maps.
    
    The maps have correlated peak positions with some offset,
    simulating the gas-to-star evolutionary sequence.
    """
    np.random.seed(seed)
    margin = 30
    
    star_image = np.zeros(shape)
    gas_image = np.zeros(shape)
    
    for i in range(n_peaks):
        # Random peak position
        y = np.random.randint(margin, shape[0] - margin)
        x = np.random.randint(margin, shape[1] - margin)
        
        # Random amplitude and size
        amp = np.random.uniform(50, 150)
        sigma = np.random.uniform(5, 10)
        
        # Add to star map
        star_image += make_gaussian(shape, y, x, amp, sigma)
        
        # Add to gas map with small offset (simulating evolution)
        offset_y = np.random.randint(-5, 6)
        offset_x = np.random.randint(-5, 6)
        gas_amp = amp * np.random.uniform(0.7, 1.3)
        gas_image += make_gaussian(shape, y + offset_y, x + offset_x, gas_amp, sigma)
    
    # Add some noise
    star_image += np.random.normal(0, 3, shape)
    gas_image += np.random.normal(0, 3, shape)
    
    return star_image, gas_image

# Create synthetic data
star_data, gas_data = create_synthetic_galaxy(n_peaks=25)
pixel_scale = 10.0  # pc per pixel

print(f"Created synthetic maps: {star_data.shape}")
print(f"Pixel scale: {pixel_scale} pc/pixel")

In [ ]:
# Visualize the synthetic data
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(star_data, origin='lower', cmap='viridis')
axes[0].set_title('Star Formation Tracer')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
plt.colorbar(im0, ax=axes[0], label='Intensity')

im1 = axes[1].imshow(gas_data, origin='lower', cmap='plasma')
axes[1].set_title('Gas Tracer')
axes[1].set_xlabel('X (pixels)')
axes[1].set_ylabel('Y (pixels)')
plt.colorbar(im1, ax=axes[1], label='Intensity')

plt.tight_layout()
plt.show()

## 3. Basic Tuningfork Analysis

Now let's run the tuningfork analysis. We need to configure the analysis parameters.

In [ ]:
# Configure the analysis
config = TuningForkConfig(
    # Aperture settings
    lap_min=30.0,           # Minimum aperture diameter (pc)
    lap_max=500.0,          # Maximum aperture diameter (pc)
    n_apertures=8,          # Number of aperture sizes
    
    # Timeline
    tstar=4.0,              # Stellar tracer timescale (Myr) - e.g., Hα
    
    # Peak detection thresholds (single values for both star and gas, like IDL)
    npixmin=20,             # Minimum pixels per peak (IDL default: 20)
    nsigma=5.0,             # Significance threshold (IDL default: 5)
    
    # Peak detection contour levels
    loglevels=True,         # Use logarithmic contour levels
    logrange_s=2.0,         # Log range for stellar contours (dex)
    logspacing_s=0.5,       # Log spacing for stellar contours (dex)
    logrange_g=2.0,         # Log range for gas contours (dex)
    logspacing_g=0.5,       # Log spacing for gas contours (dex)
    
    # Monte Carlo
    n_mc=500,               # Number of MC experiments (use 1000+ for production)
    
    # Fitting
    ndepth=3,               # Refinement depth
    ntry=51,                # Grid points per parameter
    
    # Random seed for reproducibility
    seed=42,
)

print("Configuration:")
print(f"  Apertures: {config.n_apertures} ({config.lap_min:.0f} - {config.lap_max:.0f} pc)")
print(f"  t_star: {config.tstar} Myr")
print(f"  Monte Carlo: {config.n_mc} experiments")
print(f"  Peak ID: npixmin={config.npixmin}, nsigma={config.nsigma}")
print(f"  Contour levels: loglevels={config.loglevels}, logrange_s={config.logrange_s}, logspacing_s={config.logspacing_s}")

In [ ]:
# Run the analysis
print("Running tuningfork analysis...")
result = run_tuningfork(
    star_image=star_data,
    gas_image=gas_data,
    pixel_scale=pixel_scale,
    config=config,
)
print("Analysis complete!")

## 4. Interpreting Results

Let's examine the fitted parameters and their uncertainties.

In [ ]:
# Display primary results
print("=" * 50)
print("FITTED PARAMETERS")
print("=" * 50)
print(f"t_gas   = {result.fit.tgas:.2f} (+{result.fit.tgas_errmax:.2f}/-{result.fit.tgas_errmin:.2f}) Myr")
print(f"t_over  = {result.fit.tover:.2f} (+{result.fit.tover_errmax:.2f}/-{result.fit.tover_errmin:.2f}) Myr")
print(f"lambda  = {result.fit.lambda_:.1f} (+{result.fit.lambda_errmax:.1f}/-{result.fit.lambda_errmin:.1f}) pc")
print()
print(f"chi2_min = {result.fit.chi2_min:.2f}")
print()
print(f"Detected peaks: {len(result.star_peaks)} stellar, {len(result.gas_peaks)} gas")

In [ ]:
# Calculate derived quantities
tstar = config.tstar
tgas = result.fit.tgas
tover = result.fit.tover
lambda_ = result.fit.lambda_

# Total timeline
t_total = tgas + tstar - tover

# Visibility fractions
frac_gas_only = (tgas - tover) / t_total
frac_overlap = tover / t_total
frac_star_only = (tstar - tover) / t_total

print("DERIVED QUANTITIES")
print("=" * 50)
print(f"t_total = {t_total:.2f} Myr")
print()
print("Phase fractions:")
print(f"  Gas only:   {frac_gas_only:.1%}")
print(f"  Overlap:    {frac_overlap:.1%}")
print(f"  Stars only: {frac_star_only:.1%}")

## 5. Visualization

Let's create diagnostic plots to assess the fit quality.

In [ ]:
# Plot detected peaks on the maps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Star map with peaks
axes[0].imshow(star_data, origin='lower', cmap='viridis', alpha=0.8)
if len(result.star_peaks) > 0:
    star_x = [p.x for p in result.star_peaks]
    star_y = [p.y for p in result.star_peaks]
    axes[0].scatter(star_x, star_y, c='red', s=50, marker='x', label='Detected peaks')
axes[0].set_title(f'Star Map ({len(result.star_peaks)} peaks)')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
axes[0].legend()

# Gas map with peaks
axes[1].imshow(gas_data, origin='lower', cmap='plasma', alpha=0.8)
if len(result.gas_peaks) > 0:
    gas_x = [p.x for p in result.gas_peaks]
    gas_y = [p.y for p in result.gas_peaks]
    axes[1].scatter(gas_x, gas_y, c='cyan', s=50, marker='x', label='Detected peaks')
axes[1].set_title(f'Gas Map ({len(result.gas_peaks)} peaks)')
axes[1].set_xlabel('X (pixels)')
axes[1].set_ylabel('Y (pixels)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot the tuning fork diagram
if hasattr(result, 'tuningfork_data') and result.tuningfork_data is not None:
    tf = result.tuningfork_data
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Data points with error bars
    ax.errorbar(
        tf.apertures, tf.flux_ratios,
        yerr=[tf.flux_ratio_errmin, tf.flux_ratio_errmax],
        fmt='o', color='blue', markersize=8,
        capsize=5, label='Observed'
    )
    
    # Best fit model (if available)
    if hasattr(tf, 'model_apertures') and tf.model_apertures is not None:
        ax.plot(tf.model_apertures, tf.model_flux_ratios, 
                'r-', linewidth=2, label='Best fit')
    
    ax.set_xscale('log')
    ax.set_xlabel('Aperture diameter (pc)')
    ax.set_ylabel('Flux ratio (star/gas)')
    ax.set_title('Tuning Fork Diagram')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Tuning fork data not available in result object")

In [ ]:
# Summary figure
fig = plt.figure(figsize=(14, 10))

# Timeline visualization
ax1 = fig.add_subplot(2, 2, 1)
phases = ['Gas only', 'Overlap', 'Stars only']
fractions = [frac_gas_only, frac_overlap, frac_star_only]
colors = ['blue', 'purple', 'orange']
ax1.barh(phases, fractions, color=colors)
ax1.set_xlabel('Fraction of total timeline')
ax1.set_title('Evolutionary Timeline')
ax1.set_xlim(0, 1)
for i, (phase, frac) in enumerate(zip(phases, fractions)):
    ax1.text(frac + 0.02, i, f'{frac:.1%}', va='center')

# Timescales
ax2 = fig.add_subplot(2, 2, 2)
timescales = ['$t_{gas}$', '$t_{over}$', '$t_{star}$', '$t_{total}$']
values = [tgas, tover, tstar, t_total]
errors_low = [result.fit.tgas_errmin, result.fit.tover_errmin, 0, 0]
errors_high = [result.fit.tgas_errmax, result.fit.tover_errmax, 0, 0]
ax2.barh(timescales, values, xerr=[errors_low, errors_high], 
         color=['blue', 'purple', 'orange', 'green'], capsize=5)
ax2.set_xlabel('Timescale (Myr)')
ax2.set_title('Measured Timescales')

# Peak distribution
ax3 = fig.add_subplot(2, 2, 3)
if len(result.star_peaks) > 0 and len(result.gas_peaks) > 0:
    star_flux = [p.flux for p in result.star_peaks]
    gas_flux = [p.flux for p in result.gas_peaks]
    ax3.hist(star_flux, bins=15, alpha=0.5, label='Stars', color='orange')
    ax3.hist(gas_flux, bins=15, alpha=0.5, label='Gas', color='blue')
    ax3.set_xlabel('Peak flux')
    ax3.set_ylabel('Count')
    ax3.set_title('Peak Flux Distribution')
    ax3.legend()

# Results text
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')
results_text = f"""
ANALYSIS RESULTS
{'='*40}

Fitted Parameters:
  t_gas   = {tgas:.2f} (+{result.fit.tgas_errmax:.2f}/-{result.fit.tgas_errmin:.2f}) Myr
  t_over  = {tover:.2f} (+{result.fit.tover_errmax:.2f}/-{result.fit.tover_errmin:.2f}) Myr
  lambda  = {lambda_:.1f} (+{result.fit.lambda_errmax:.1f}/-{result.fit.lambda_errmin:.1f}) pc

Quality:
  chi2_min = {result.fit.chi2_min:.2f}

Statistics:
  Star peaks: {len(result.star_peaks)}
  Gas peaks:  {len(result.gas_peaks)}
"""
ax4.text(0.1, 0.9, results_text, transform=ax4.transAxes,
         fontfamily='monospace', fontsize=11, verticalalignment='top')

plt.suptitle('Heisenberg Analysis Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Saving Results

Export results to JSON and text formats.

In [ ]:
# Save to files
output_dir = Path('.')
galaxy_name = 'demo_galaxy'

# JSON output (full results)
json_path = output_dir / f'{galaxy_name}_results.json'
write_results(result, json_path)
print(f"Results saved to: {json_path}")

# Text summary
txt_path = output_dir / f'{galaxy_name}_summary.txt'
write_summary(result, txt_path)
print(f"Summary saved to: {txt_path}")

In [ ]:
# Read back and display
import json

with open(json_path) as f:
    loaded_results = json.load(f)

print("Loaded results structure:")
for key in loaded_results.keys():
    print(f"  - {key}")

## 7. Parameter Sensitivity

Let's explore how results change with different parameter choices.

In [ ]:
# Test different npixmin values
npixmin_values = [10, 15, 20, 30]
results_by_npixmin = []

print("Testing different npixmin values...")
for npixmin in npixmin_values:
    test_config = TuningForkConfig(
        lap_min=30.0, lap_max=500.0, n_apertures=8,
        tstar=4.0,
        npixmin=npixmin,        # Single npixmin for both star and gas (like IDL)
        nsigma=5.0,             # Single nsigma for both star and gas (like IDL)
        n_mc=200,  # Fewer MC for speed
        ndepth=2, ntry=31,
        seed=42,
    )
    
    res = run_tuningfork(star_data, gas_data, pixel_scale, test_config)
    results_by_npixmin.append({
        'npixmin': npixmin,
        'n_star_peaks': len(res.star_peaks),
        'n_gas_peaks': len(res.gas_peaks),
        'tgas': res.fit.tgas,
        'lambda': res.fit.lambda_,
    })
    print(f"  npixmin={npixmin}: {len(res.star_peaks)} star peaks, {len(res.gas_peaks)} gas peaks")

print("Done!")

In [ ]:
# Visualize parameter sensitivity
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

npix = [r['npixmin'] for r in results_by_npixmin]

# Peak counts
axes[0].plot(npix, [r['n_star_peaks'] for r in results_by_npixmin], 'o-', label='Stars')
axes[0].plot(npix, [r['n_gas_peaks'] for r in results_by_npixmin], 's-', label='Gas')
axes[0].set_xlabel('npixmin')
axes[0].set_ylabel('Number of peaks')
axes[0].set_title('Peak Count vs npixmin')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# t_gas
axes[1].plot(npix, [r['tgas'] for r in results_by_npixmin], 'o-', color='blue')
axes[1].set_xlabel('npixmin')
axes[1].set_ylabel('t_gas (Myr)')
axes[1].set_title('t_gas vs npixmin')
axes[1].grid(True, alpha=0.3)

# lambda
axes[2].plot(npix, [r['lambda'] for r in results_by_npixmin], 'o-', color='green')
axes[2].set_xlabel('npixmin')
axes[2].set_ylabel('lambda (pc)')
axes[2].set_title('lambda vs npixmin')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Next Steps

This demo covered the basics of Heisenberg analysis. For more advanced usage:

1. **Real data**: Load FITS files using `heisenberg.io.read_fits()`
2. **Diffuse filtering**: Use `run_diffuse_iteration()` for iterative analysis
3. **Masking**: Apply DS9 region masks using `heisenberg.imaging.mask_tool`
4. **CLI**: Use `heisenberg run config.yaml` for command-line analysis

See the documentation for more details:
- [Quick Start Guide](../../docs/quickstart.md)
- [Configuration Reference](../../docs/python/configuration.md)
- [Parameter Tuning](../../docs/tutorials/parameter-tuning.md)